# Test Fitting Plot — SLP MNIST

For a chosen `(PRUNING, BATCH_SIZE, N_RUNS)`, this notebook produces a single graph with three layers:

| Layer | Style | What it shows |
|-------|-------|---------------|
| Raw data | Dim scatter (dots) | Actual CE_Test values from each of the N runs |
| Individual fits | Dim lines | Fitted curve `CE(x) = A + B/(x+1)^n` for each run |
| Average fit | Solid bold line + shading | Point-wise mean ± 1 std of all fitted curves |

The asymptote value `<A>` (mean of all fitted A parameters) is shown as a horizontal dashed line with a label.

A data CSV is saved to this directory alongside the notebook.

---
**Variables to change:** `PRUNING`, `BATCH_SIZE`, `N_RUNS`, `COLOR` in Cell 1.

## Cell 1 — Configuration

All user-facing knobs live here. Change these and re-run all cells.

- **`PRUNING`** — pruning fraction (e.g. `0.0`, `0.5`, `0.8`, `0.9`)
- **`BATCH_SIZE`** — one of `64`, `1024`, `60000`
- **`N_RUNS`** — how many runs to include (1–100; use `100` for all)
- **`COLOR`** — color for the **average** curve and its asymptote line; individual runs are auto-colored from `plt.cm.turbo`
- **`COLORMAP`** — matplotlib colormap name used for per-run colors (default `"turbo"`; try `"tab20"`, `"rainbow"`, `"plasma"`)

In [2]:
!pip install lmfit

In [19]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from lmfit import Parameters, minimize
import warnings
warnings.filterwarnings("ignore")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  USER VARIABLES  ← change these
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PRUNING    = 0.0        # pruning fraction: 0.0, 0.1, ..., 0.8, 0.82, ..., 1.0
BATCH_SIZE = 64         # 64 | 1024 | 60000
N_RUNS     = 3         # how many runs to show (1-100)
COLOR      = "black"    # color for the average curve + its asymptote line
COLORMAP   = "turbo"    # colormap for per-run colors: "turbo" | "tab20" | "rainbow" | "plasma"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
OUT_DIR  = os.path.dirname(os.path.abspath("__file__"))  # same folder as this notebook

DATA_DIR = os.path.join(BASE_DIR, f"p-percentage_{PRUNING}", f"batch_size_{BATCH_SIZE}")

# ── Fitting bounds (match fitting_function_IPA.ipynb) ─────────────────────────
A_MIN, A_MAX = 0.1, 2.3
B_MIN, B_MAX = 0.0, 1000.0
N_MIN, N_MAX = 0.5, 3.0

LN10 = np.log(10)

# ── Verify data directory exists ──────────────────────────────────────────────
assert os.path.isdir(DATA_DIR), f"Data directory not found:\n  {DATA_DIR}"

all_files = sorted(glob.glob(os.path.join(DATA_DIR, f"slp_{PRUNING}_{BATCH_SIZE}_run_*.txt")))
assert all_files, f"No run files found in {DATA_DIR}"

files = all_files[:N_RUNS]   # take the first N_RUNS files

print(f"Pruning  : {PRUNING*100:.1f}%")
print(f"Batch    : {BATCH_SIZE}")
print(f"Runs     : {len(files)} (of {len(all_files)} available)")
print(f"Avg color: {COLOR}")
print(f"Colormap : {COLORMAP}")
print(f"Data dir : {DATA_DIR}")

Pruning  : 0.0%
Batch    : 64
Runs     : 3 (of 100 available)
Avg color: black
Colormap : turbo
Data dir : C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL\p-percentage_0.0\batch_size_64


## Cell 2 — Load Raw Data & Fit Each Run

For each of the `N_RUNS` run files:
1. Reads `CE_TEST` and `Batch_Number` columns
2. Fits `CE(x) = A + B / (x+1)^n` using `lmfit` with weighted residuals (same settings as `fitting_function_IPA.ipynb`)
3. Stores raw scatter data and fitted parameters

Runs that fail to load or converge are skipped and reported.

In [20]:
# ── Fitting helpers (identical to fitting_function_IPA.ipynb) ─────────────────

def initialize_guesses(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    A0 = np.percentile(y, 5)
    B0 = np.percentile(y, 95) - A0
    n0 = 0.5
    if len(x) > 10:
        denom = y[0] - A0
        if abs(denom) > 1e-10:
            frac = max(1e-6, (y[0] - y[-1]) / denom)
            if frac > 0:
                n0 = max(0.3, min(1.5, -np.log(frac)))
    return A0, n0, B0

def _model(params, x):
    v = params.valuesdict()
    return v['A'] + v['B'] / ((x + 1) ** v['n'])

def _residual(params, x, data):
    return x * (_model(params, x) - data)  # weight = x

def fit_run(x, y):
    """Returns (A, B, n) or None on failure."""
    mask = ~np.isnan(y)
    x, y = x[mask], y[mask]
    if len(x) < 10:
        return None
    A0, n0, B0 = initialize_guesses(x, y)
    params = Parameters()
    params.add('A', value=A0, min=A_MIN, max=A_MAX)
    params.add('B', value=B0, min=B_MIN, max=B_MAX)
    params.add('n', value=n0, min=N_MIN, max=N_MAX)
    try:
        result = minimize(_residual, params, args=(x, y))
        v = result.params.valuesdict()
        return v['A'], v['B'], v['n']
    except Exception:
        return None


# ── Load & fit ────────────────────────────────────────────────────────────────

raw_data   = []   # list of DataFrames with Batch_Number, CE_TEST, run_id
fit_params = []   # list of (run_id, A, B, n)
skipped    = []

for fpath in files:
    run_id = os.path.basename(fpath)

    try:
        df = pd.read_csv(fpath, sep=r"\s+")
        df.columns = df.columns.str.strip()
        df["CE_TEST"]      = pd.to_numeric(df["CE_TEST"],      errors="coerce")
        df["Batch_Number"] = pd.to_numeric(df["Batch_Number"], errors="coerce")
        df = df.dropna(subset=["CE_TEST", "Batch_Number"])
        if len(df) < 5:
            skipped.append((run_id, "too few rows"))
            continue
    except Exception as e:
        skipped.append((run_id, str(e)))
        continue

    x = df["Batch_Number"].values
    y = df["CE_TEST"].values

    result = fit_run(x, y)
    if result is None:
        skipped.append((run_id, "fit did not converge"))
        continue

    A, B, n = result
    raw_df = df[["Batch_Number", "CE_TEST"]].copy()
    raw_df["run_id"] = run_id
    raw_data.append(raw_df)
    fit_params.append({"run_id": run_id, "A": A, "B": B, "n": n})

n_ok = len(fit_params)
print(f"Successfully loaded & fitted : {n_ok} / {len(files)} runs")
if skipped:
    print(f"Skipped ({len(skipped)}):")
    for name, reason in skipped:
        print(f"  {name}  —  {reason}")

Successfully loaded & fitted : 3 / 3 runs


## Cell 3 — Compute Average Fitted Curve & Save Data CSV

1. Builds a **union x-grid** from all run batch numbers
2. Re-evaluates every fitted `(A, B, n)` on that grid → stacks into a matrix
3. Takes point-wise mean and std → the average curve
4. Saves everything to `test_fitting_data_p{PRUNING}_bs{BATCH_SIZE}_n{N_RUNS}.csv`

**CSV columns:**
- `run_id` — source filename (blank for average rows)
- `type` — `raw_scatter` | `fitted_curve` | `average_curve`
- `Batch_Number` — x value
- `CE_value` — y value
- `A`, `B`, `n` — fitted parameters for that run (NaN for average rows)
- `mean_A`, `std_A` — summary stats (same value repeated on every row for easy access)

In [21]:
assert fit_params, "No successfully fitted runs — cannot compute average."

params_df = pd.DataFrame(fit_params)  # columns: run_id, A, B, n

# ── Union x-grid from all raw batch numbers ───────────────────────────────────
all_x = np.sort(np.unique(np.concatenate([d["Batch_Number"].values for d in raw_data])))

# ── Re-evaluate every fit on the union grid ───────────────────────────────────
fitted_matrix = np.array([
    row["A"] + row["B"] / ((all_x + 1) ** row["n"])
    for _, row in params_df.iterrows()
])  # shape: (n_runs, len(all_x))

mean_curve = fitted_matrix.mean(axis=0)
std_curve  = fitted_matrix.std(axis=0)

mean_A = params_df["A"].mean()
std_A  = params_df["A"].std()
mean_B = params_df["B"].mean()
mean_n = params_df["n"].mean()

print(f"Union x-grid : {len(all_x)} points  [{all_x[0]:.0f} → {all_x[-1]:.0f}]")
print(f"mean A (asymptote) : {mean_A:.4f}  ±  {std_A:.4f}")
print(f"mean B             : {mean_B:.4f}")
print(f"mean n             : {mean_n:.4f}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Save data CSV
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
csv_rows = []

# Raw scatter rows
for d in raw_data:
    run_id = d["run_id"].iloc[0]
    p_row  = params_df[params_df["run_id"] == run_id].iloc[0]
    for _, r in d.iterrows():
        csv_rows.append({
            "run_id": run_id, "type": "raw_scatter",
            "Batch_Number": r["Batch_Number"], "CE_value": r["CE_TEST"],
            "A": p_row["A"], "B": p_row["B"], "n": p_row["n"],
            "mean_A": mean_A, "std_A": std_A
        })

# Fitted curve rows (evaluated on original run x-grid)
for d, (_, p_row) in zip(raw_data, params_df.iterrows()):
    x_run  = d["Batch_Number"].values
    y_fit  = p_row["A"] + p_row["B"] / ((x_run + 1) ** p_row["n"])
    for xi, yi in zip(x_run, y_fit):
        csv_rows.append({
            "run_id": p_row["run_id"], "type": "fitted_curve",
            "Batch_Number": xi, "CE_value": yi,
            "A": p_row["A"], "B": p_row["B"], "n": p_row["n"],
            "mean_A": mean_A, "std_A": std_A
        })

# Average curve rows
for xi, yi_mean, yi_std in zip(all_x, mean_curve, std_curve):
    csv_rows.append({
        "run_id": "average", "type": "average_curve",
        "Batch_Number": xi, "CE_value": yi_mean,
        "A": np.nan, "B": np.nan, "n": np.nan,
        "mean_A": mean_A, "std_A": std_A
    })

csv_df   = pd.DataFrame(csv_rows)
csv_name = f"test_fitting_data_p{PRUNING}_bs{BATCH_SIZE}_n{n_ok}.csv"
csv_path = os.path.join(OUT_DIR, csv_name)
csv_df.to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}  ({len(csv_df):,} rows)")

Union x-grid : 200 points  [0 → 199]
mean A (asymptote) : 0.3038  ±  0.0054
mean B             : 6.3294
mean n             : 0.9376

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\TEST_FITTING\test_fitting_data_p0.0_bs64_n3.csv  (1,240 rows)


## Cell 4 — Plot

Single figure with three layers on the same axes. Each run gets its own color drawn from `COLORMAP`.

| Layer | Style | Alpha | Color |
|-------|-------|-------|-------|
| Raw CE_Test values | Dim scatter (dots) | 0.20 | Per-run (colormap) |
| Individual fitted curves | Dim lines | 0.30 | Per-run (colormap) |
| Per-run asymptote | Small dot + label at curve end | 0.75 | Per-run (colormap) |
| Average fitted curve | Solid bold line | 1.0 | `COLOR` |
| ±1 std band | Filled area | 0.15 | `COLOR` |
| Average asymptote | Dashed line + label | 0.85 | `COLOR` |

In [ ]:
# -- Style matching fitting_function_IPA.ipynb --------------------------------
plt.rcParams.update({
    "font.size":        18,
    "xtick.labelsize":  16,
    "ytick.labelsize":  16,
    "legend.fontsize":  14,
})

# -- Per-run color palette ----------------------------------------------------
cmap       = plt.get_cmap(COLORMAP)
run_colors = [cmap(i / max(n_ok - 1, 1)) for i in range(n_ok)]

fig, ax = plt.subplots(figsize=(14, 6))

# -- Individual runs: scatter + fitted curve + per-run asymptote label --------
for i, (d, (_, p_row)) in enumerate(zip(raw_data, params_df.iterrows())):
    x_run   = d["Batch_Number"].values
    y_fit   = p_row["A"] + p_row["B"] / ((x_run + 1) ** p_row["n"])
    A_i     = p_row["A"]
    c       = run_colors[i]
    run_num = os.path.basename(p_row["run_id"]).split("_run_")[-1].replace(".txt", "")

    # dim scatter: raw CE_Test data points
    ax.scatter(x_run, d["CE_TEST"], color=c, s=6, alpha=0.20, linewidths=0, zorder=1)

    # dim fitted curve
    ax.plot(x_run, y_fit, color=c, linewidth=1.5, alpha=0.40, zorder=2,
            label=f"Run {run_num}  A={A_i:.3f}")

    # asymptote: small dot at end of curve + label
    ax.scatter(x_run[-1], A_i, color=c, s=30, alpha=0.85, zorder=4, linewidths=0)
    ax.text(
        x_run[-1] + max(all_x) * 0.006, A_i,
        f"A={A_i:.3f}",
        color=c, fontsize=12, alpha=0.90, va="center", ha="left", zorder=5
    )

# -- v4 averaged curve (raw average, no curve fitting) ------------------------
if V4_DF is not None:
    x_v4 = V4_DF[v4_bn_col].values.astype(float)
    y_v4 = np.minimum(V4_DF[v4_ce_col].values.astype(float), LN10)
    ax.plot(x_v4, y_v4,
            color=V4_COLOR, linewidth=2.2, linestyle="--", alpha=0.85, zorder=6,
            label="v4 avg (raw average)")

# -- Average of N fitted curves + 1-std band ----------------------------------
ax.fill_between(all_x, mean_curve - std_curve, mean_curve + std_curve,
                color=AVG_COLOR, alpha=0.12, zorder=5)
ax.plot(all_x, mean_curve,
        color=AVG_COLOR, linewidth=2.8, zorder=7,
        label=f"Avg of {n_ok} fitted curves   <A>={mean_A:.3f}")

# -- Average asymptote dashed line + label ------------------------------------
ax.axhline(mean_A, color=AVG_COLOR, linewidth=1.2, linestyle="--", alpha=0.70, zorder=6)
ax.text(
    all_x[int(len(all_x) * 0.02)], mean_A + 0.06,
    f"<A> = {mean_A:.4f}  (+/-{std_A:.4f})",
    color=AVG_COLOR, fontsize=13, fontweight="bold", va="bottom"
)

# -- ln(10) reference (matching fitting_function_IPA.ipynb placement) ---------
ax.text(0.06, LN10 + 0.05, r"$\ln(10)$",
        transform=ax.get_yaxis_transform(), fontsize=14, va="center")

# -- Title inside axes (matching fitting_function_IPA.ipynb) ------------------
ax.text(0.5, 0.95, "Cross-Entropy",
        transform=ax.transAxes, ha="center", va="top")
ax.text(0.5, 0.89, f"(SLP MNIST  |  P%={PRUNING*100:.1f}  |  BS={BATCH_SIZE})",
        transform=ax.transAxes, ha="center", va="top", fontsize=16)

# -- Axes ---------------------------------------------------------------------
ax.set_xlabel("Batch Number")
ax.set_ylabel("Average CE")
ax.set_ylim(0, 2.7)
ax.grid(True)

# -- Legend outside right (matching fitting_function_IPA.ipynb) ---------------
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles, labels,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    borderaxespad=0,
)

plt.tight_layout()

# -- Save ---------------------------------------------------------------------
fig_name = f"test_fitting_plot_p{PRUNING}_bs{BATCH_SIZE}_n{n_ok}.png"
fig_path = os.path.join(OUT_DIR, fig_name)
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved: {fig_path}")
plt.show()